# ROGII Wellbore Geology Prediction - Simple Baseline

This notebook builds a first reliable Kaggle baseline for predicting `tvt` on the hidden evaluation rows of horizontal wells.

Main rules used here:

- The target is `TVT`, submitted as `tvt`.
- The public score is RMSE, so validation also uses RMSE.
- Validation is grouped by `well`, because rows from the same well are highly related.
- Training uses rows where `TVT_input` is missing, because those rows mimic the hidden test evaluation zone.
- We never train on test labels and we do not use external data.

## 1. Imports and Paths

Kaggle usually has `pandas`, `numpy`, `scikit-learn`, and `joblib` installed already. The notebook searches for the folder that contains `train/`, `test/`, and `sample_submission.csv`, then writes outputs to a writable working folder.

In [1]:
from pathlib import Path
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)


# If automatic discovery fails, set this manually.
# Local Windows example:
# DATA_ROOT_OVERRIDE = Path(r"C:\Users\alem\Documents\codes\kaggle_rogii")
# Kaggle example:
# DATA_ROOT_OVERRIDE = Path("/kaggle/input/your-dataset-folder")
DATA_ROOT_OVERRIDE = None


def describe_available_inputs() -> str:
    """Return a compact listing of likely data folders to debug Kaggle input paths."""
    lines = [f"Current working directory: {Path.cwd()}"]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        lines.append("/kaggle/input contents:")
        for path in sorted(kaggle_input.iterdir()):
            if path.is_dir():
                child_names = ", ".join(sorted(p.name for p in path.iterdir())[:8])
                lines.append(f"  - {path} -> {child_names}")
            else:
                lines.append(f"  - {path}")
    else:
        lines.append("/kaggle/input does not exist in this environment.")
    return "\n".join(lines)


def is_data_root(path: Path) -> bool:
    """Check whether a path has the exact competition file structure."""
    return (path / "train").exists() and (path / "test").exists() and (path / "sample_submission.csv").exists()


def safe_sample_submission_parents(root: Path):
    """Find sample_submission.csv below safe roots without crashing on system folders."""
    try:
        yield from (path.parent for path in root.rglob("sample_submission.csv"))
    except OSError as error:
        print(f"Skipping recursive search under {root}: {error}")


def find_data_root() -> Path:
    """Find the folder containing train/, test/, and sample_submission.csv."""
    if DATA_ROOT_OVERRIDE is not None:
        path = Path(DATA_ROOT_OVERRIDE)
        if (path / "train").exists() and (path / "test").exists() and (path / "sample_submission.csv").exists():
            return path
        print(f"Warning: DATA_ROOT_OVERRIDE does not exist in this kernel: {path}")
        print("If this is a Windows C:\\ path, it will not work inside a Kaggle/Linux notebook.")
        print("Continuing with automatic discovery...\n")

    local_windows_root = Path(r"C:\Users\alem\Documents\codes\kaggle_rogii")
    search_roots = [Path.cwd(), Path.cwd().parent, local_windows_root]
    if Path("/kaggle/input").exists():
        search_roots.append(Path("/kaggle/input"))
    checked = []

    for root in search_roots:
        if not root.exists():
            continue

        # First check the root itself.
        candidate_roots = [root]

        # Recursive search is safe inside the working folder and Kaggle input, but not from broad
        # Linux roots such as / or /tmp because those can contain /proc and other virtual filesystems.
        if root == Path.cwd() or str(root).startswith("/kaggle/input"):
            candidate_roots.extend(safe_sample_submission_parents(root))

        for path in dict.fromkeys(candidate_roots):
            checked.append(str(path))
            if is_data_root(path):
                return path

    message = "Could not find a folder containing train/, test/, and sample_submission.csv.\n"
    message += describe_available_inputs()
    message += "\n\nChecked roots include:\n" + "\n".join(checked[:30])
    raise FileNotFoundError(message)


DATA_ROOT = find_data_root()

# Kaggle input folders are read-only, so outputs go to /kaggle/working when available.
OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
MODEL_DIR = OUTPUT_ROOT / "models"
SUBMISSION_DIR = OUTPUT_ROOT / "submissions"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"

print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)

Data root: C:\Users\alem\Documents\codes\kaggle_rogii
Output root: C:\Users\alem\Documents\codes\kaggle_rogii


## 2. Inspect the Data Structure

Each training well has a horizontal well CSV, a typewell CSV, and usually a PNG. The hidden test set will contain different wells, but the submission format is always defined by `sample_submission.csv`.

In [2]:
train_horizontal_files = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
train_typewell_files = sorted(TRAIN_DIR.glob("*__typewell.csv"))
test_horizontal_files = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
test_typewell_files = sorted(TEST_DIR.glob("*__typewell.csv"))
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Training horizontal wells: {len(train_horizontal_files)}")
print(f"Training typewells:        {len(train_typewell_files)}")
print(f"Visible test wells:        {len(test_horizontal_files)}")
print(f"Visible test typewells:    {len(test_typewell_files)}")
print(f"Sample submission shape:   {sample_submission.shape}")

display(sample_submission.head())

Training horizontal wells: 773
Training typewells:        773
Visible test wells:        3
Visible test typewells:    3
Sample submission shape:   (14151, 2)


,id,tvt
0,000d7d20_1442,0.0
1,000d7d20_1443,0.0
2,000d7d20_1444,0.0
3,000d7d20_1445,0.0
4,000d7d20_1446,0.0


In [3]:
# Read one example well so we can see the columns and missing values.
example_well = train_horizontal_files[0].name.split("__")[0]
example_horizontal = pd.read_csv(TRAIN_DIR / f"{example_well}__horizontal_well.csv")
example_typewell = pd.read_csv(TRAIN_DIR / f"{example_well}__typewell.csv")

print("Example well:", example_well)
print("Horizontal shape:", example_horizontal.shape)
display(example_horizontal.head())
display(example_horizontal.isna().sum().to_frame("missing_rows"))

print("Typewell shape:", example_typewell.shape)
display(example_typewell.head())
display(example_typewell.isna().sum().to_frame("missing_rows"))

Example well: 000d7d20
Horizontal shape: (5278, 13)


,MD,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input
0,11467.0,2983525.16,1069022.09,-9258.57,-9395.81,-9569.86,-9597.64,-9670.99,-9705.96,-9846.35,11236.02,115.692586,11236.02
1,11468.0,2983525.18,1069022.30,-9259.55,-9395.75,-9569.80,-9597.58,-9670.93,-9705.90,-9846.29,11237.05,115.584293,11237.05
2,11469.0,2983525.20,1069022.52,-9260.52,-9395.69,-9569.74,-9597.52,-9670.87,-9705.84,-9846.23,11238.09,135.446960,11238.09
3,11470.0,2983525.22,1069022.73,-9261.50,-9395.64,-9569.69,-9597.47,-9670.82,-9705.79,-9846.18,11239.12,140.401346,11239.12
4,11471.0,2983525.25,1069022.95,-9262.47,-9395.58,-9569.63,-9597.41,-9670.76,-9705.73,-9846.12,11240.15,111.270638,11240.15


,missing_rows
MD,0
X,0
Y,0
Z,0
ANCC,0
ASTNU,0
ASTNL,0
EGFDU,0
EGFDL,0
BUDA,0


Typewell shape: (1296, 3)


,TVT,GR,Geology
0,11223.95,126.11,NaN
1,11224.45,128.22,NaN
2,11224.95,128.72,NaN
3,11225.45,128.12,NaN
4,11225.95,125.29,NaN


,missing_rows
TVT,0
GR,0
Geology,299


## 3. What We Need to Predict

For every `id` in `sample_submission.csv`, we must predict `tvt`, which is the true vertical thickness/geological position for one row of a horizontal well.

The `id` has the format `{well}_{row_index}`. For example, `000d7d20_1442` means row index `1442` in well `000d7d20`. In training, the answer is the `TVT` column. In test, those target values are hidden in the evaluation zone, and `TVT_input` is `NaN` for those rows.

## 4. Load Data and Build Non-Leaky Features

`TVT_input` is dangerous: outside the evaluation zone it is a copy of `TVT`, so using those rows directly as training examples would leak the target. This baseline trains only on rows where `TVT_input` is missing, which is the same condition as the rows we must submit for.

We still use safe features derived from the provided `TVT_input` anchors around the missing zone, such as forward fill, backward fill, and linear interpolation. These are available in both train and test files and do not use hidden labels.

In [4]:
def get_well_name(path: Path) -> str:
    """Extract the 8-character well id from a file name."""
    return path.name.split("__")[0]


def summarize_typewell(typewell_path: Path) -> dict:
    """Create simple per-well summary features from the vertical reference log."""
    typewell = pd.read_csv(typewell_path)
    geology = typewell.get("Geology", pd.Series(dtype="object")).astype("object")
    geology_mode = geology.dropna().mode()

    # Stable small numeric code for the most common geology label.
    # Python's built-in hash is intentionally randomized between sessions, so avoid it here.
    geology_mode_code = sum(ord(ch) for ch in str(geology_mode.iloc[0])) if len(geology_mode) else -1

    return {
        "typewell_rows": len(typewell),
        "typewell_tvt_min": typewell["TVT"].min(),
        "typewell_tvt_max": typewell["TVT"].max(),
        "typewell_tvt_range": typewell["TVT"].max() - typewell["TVT"].min(),
        "typewell_gr_mean": typewell["GR"].mean(),
        "typewell_gr_std": typewell["GR"].std(),
        "typewell_gr_min": typewell["GR"].min(),
        "typewell_gr_max": typewell["GR"].max(),
        "typewell_geology_unique": geology.nunique(dropna=True),
        "typewell_geology_mode_code": geology_mode_code,
    }


def add_horizontal_features(horizontal: pd.DataFrame, well: str, typewell_features: dict) -> pd.DataFrame:
    """Add row-level and well-level features available for both train and test."""
    df = horizontal.copy()
    df["well"] = well
    df["row_index"] = np.arange(len(df))
    df["id"] = well + "_" + df["row_index"].astype(str)

    # Relative coordinates help the model learn local well shape instead of memorizing absolute positions.
    for col in ["MD", "X", "Y", "Z"]:
        if col in df.columns:
            df[f"{col.lower()}_rel"] = df[col] - df[col].iloc[0]

    # Gamma ray context: local averages and local change along the wellbore.
    if "GR" in df.columns:
        df["gr_roll_mean_5"] = df["GR"].rolling(window=5, min_periods=1, center=True).mean()
        df["gr_roll_mean_25"] = df["GR"].rolling(window=25, min_periods=1, center=True).mean()
        df["gr_diff_1"] = df["GR"].diff().fillna(0)

    # Features based only on provided TVT_input values. Evaluation rows have TVT_input missing.
    if "TVT_input" in df.columns:
        known = df["TVT_input"].notna()
        idx = pd.Series(np.arange(len(df)), index=df.index)
        prev_known_idx = idx.where(known).ffill()
        next_known_idx = idx.where(known).bfill()

        df["tvt_input_is_missing"] = df["TVT_input"].isna().astype(int)
        df["tvt_input_ffill"] = df["TVT_input"].ffill()
        df["tvt_input_bfill"] = df["TVT_input"].bfill()
        df["tvt_input_linear"] = df["TVT_input"].interpolate(method="linear", limit_direction="both")
        df["dist_prev_tvt_input"] = idx - prev_known_idx
        df["dist_next_tvt_input"] = next_known_idx - idx

    for name, value in typewell_features.items():
        df[name] = value

    return df


def load_folder(folder: Path, has_target: bool) -> pd.DataFrame:
    """Load all wells from a folder and return one modeling table."""
    frames = []
    for horizontal_path in sorted(folder.glob("*__horizontal_well.csv")):
        well = get_well_name(horizontal_path)
        typewell_path = folder / f"{well}__typewell.csv"

        horizontal = pd.read_csv(horizontal_path)
        typewell_features = summarize_typewell(typewell_path)
        frame = add_horizontal_features(horizontal, well, typewell_features)

        if has_target and "TVT" not in frame.columns:
            raise ValueError(f"Training file is missing TVT target: {horizontal_path}")

        frames.append(frame)

    return pd.concat(frames, ignore_index=True)


train_df = load_folder(TRAIN_DIR, has_target=True)
test_df = load_folder(TEST_DIR, has_target=False)

print("Train table:", train_df.shape)
print("Test table:", test_df.shape)
print("Rows with missing TVT_input in train:", train_df["TVT_input"].isna().sum())
print("Rows with missing TVT_input in visible test:", test_df["TVT_input"].isna().sum())
display(train_df.head())

Train table: (5092255, 39)
Test table: (19221, 32)
Rows with missing TVT_input in train: 3783989
Rows with missing TVT_input in visible test: 14151


,MD,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input,well,row_index,id,md_rel,x_rel,y_rel,z_rel,gr_roll_mean_5,gr_roll_mean_25,gr_diff_1,tvt_input_is_missing,tvt_input_ffill,tvt_input_bfill,tvt_input_linear,dist_prev_tvt_input,dist_next_tvt_input,typewell_rows,typewell_tvt_min,typewell_tvt_max,typewell_tvt_range,typewell_gr_mean,typewell_gr_std,typewell_gr_min,typewell_gr_max,typewell_geology_unique,typewell_geology_mode_code
0,11467.0,2983525.16,1069022.09,-9258.57,-9395.81,-9569.86,-9597.64,-9670.99,-9705.96,-9846.35,11236.02,115.692586,11236.02,000d7d20,0,000d7d20_0,0.0,0.00,0.00,0.00,122.241280,125.843628,0.000000,0,11236.02,11236.02,11236.02,0.0,0.0,1296,11223.95,11871.45,647.5,83.257639,26.251321,28.66,158.18,10,277
1,11468.0,2983525.18,1069022.30,-9259.55,-9395.75,-9569.80,-9597.58,-9670.93,-9705.90,-9846.29,11237.05,115.584293,11237.05,000d7d20,1,000d7d20_1,1.0,0.02,0.21,-0.98,126.781296,125.763152,-0.108293,0,11237.05,11237.05,11237.05,0.0,0.0,1296,11223.95,11871.45,647.5,83.257639,26.251321,28.66,158.18,10,277
2,11469.0,2983525.20,1069022.52,-9260.52,-9395.69,-9569.74,-9597.52,-9670.87,-9705.84,-9846.23,11238.09,135.446960,11238.09,000d7d20,2,000d7d20_2,2.0,0.04,0.43,-1.95,123.679165,125.556236,19.862666,0,11238.09,11238.09,11238.09,0.0,0.0,1296,11223.95,11871.45,647.5,83.257639,26.251321,28.66,158.18,10,277
3,11470.0,2983525.22,1069022.73,-9261.50,-9395.64,-9569.69,-9597.47,-9670.82,-9705.79,-9846.18,11239.12,140.401346,11239.12,000d7d20,3,000d7d20_3,3.0,0.06,0.64,-2.93,122.296629,125.503782,4.954386,0,11239.12,11239.12,11239.12,0.0,0.0,1296,11223.95,11871.45,647.5,83.257639,26.251321,28.66,158.18,10,277
4,11471.0,2983525.25,1069022.95,-9262.47,-9395.58,-9569.63,-9597.41,-9670.76,-9705.73,-9846.12,11240.15,111.270638,11240.15,000d7d20,4,000d7d20_4,4.0,0.09,0.86,-3.90,122.047556,125.731946,-29.130707,0,11240.15,11240.15,11240.15,0.0,0.0,1296,11223.95,11871.45,647.5,83.257639,26.251321,28.66,158.18,10,277


## 5. Validation Strategy

Rows from one well are not independent. A random row split would put nearby points from the same well in both train and validation and would make the score too optimistic. We use `GroupKFold` with `well` as the group, so whole wells are held out together.

The validation rows are also restricted to `TVT_input.isna()`, matching the rows that need predictions in the submission.

In [5]:
TARGET = "TVT"

# Use columns that exist in both train and test after feature engineering.
CANDIDATE_FEATURES = [
    "MD", "X", "Y", "Z", "GR",
    "row_index", "md_rel", "x_rel", "y_rel", "z_rel",
    "gr_roll_mean_5", "gr_roll_mean_25", "gr_diff_1",
    "tvt_input_is_missing", "tvt_input_ffill", "tvt_input_bfill", "tvt_input_linear",
    "dist_prev_tvt_input", "dist_next_tvt_input",
    "typewell_rows", "typewell_tvt_min", "typewell_tvt_max", "typewell_tvt_range",
    "typewell_gr_mean", "typewell_gr_std", "typewell_gr_min", "typewell_gr_max",
    "typewell_geology_unique", "typewell_geology_mode_code",
]

FEATURES = [col for col in CANDIDATE_FEATURES if col in train_df.columns and col in test_df.columns]
print(f"Using {len(FEATURES)} features")
print(FEATURES)

# Training rows must have a real target and should mimic hidden evaluation rows.
train_mask = train_df[TARGET].notna() & train_df["TVT_input"].isna()

if train_mask.sum() == 0:
    # This fallback keeps the notebook runnable if the local sample differs from the final competition files.
    # We still avoid raw TVT_input as a direct feature by relying on derived context features.
    print("Warning: no missing TVT_input rows found in train; using all labeled rows as a fallback.")
    train_mask = train_df[TARGET].notna()

model_df = train_df.loc[train_mask].reset_index(drop=True)
X = model_df[FEATURES]
y = model_df[TARGET]
groups = model_df["well"]

print("Modeling rows:", len(model_df))
print("Modeling wells:", groups.nunique())
display(model_df[["well", "id", TARGET, "TVT_input"] + FEATURES[:8]].head())

Using 29 features
['MD', 'X', 'Y', 'Z', 'GR', 'row_index', 'md_rel', 'x_rel', 'y_rel', 'z_rel', 'gr_roll_mean_5', 'gr_roll_mean_25', 'gr_diff_1', 'tvt_input_is_missing', 'tvt_input_ffill', 'tvt_input_bfill', 'tvt_input_linear', 'dist_prev_tvt_input', 'dist_next_tvt_input', 'typewell_rows', 'typewell_tvt_min', 'typewell_tvt_max', 'typewell_tvt_range', 'typewell_gr_mean', 'typewell_gr_std', 'typewell_gr_min', 'typewell_gr_max', 'typewell_geology_unique', 'typewell_geology_mode_code']
Modeling rows: 3783989
Modeling wells: 773


,well,id,TVT,TVT_input,MD,X,Y,Z,GR,row_index,md_rel,x_rel
0,000d7d20,000d7d20_1442,11747.38,NaN,12909.0,2983537.06,1070212.72,-9735.07,NaN,1442,1442.0,11.90
1,000d7d20,000d7d20_1443,11747.39,NaN,12910.0,2983537.03,1070213.72,-9735.06,NaN,1443,1443.0,11.87
2,000d7d20,000d7d20_1444,11747.40,NaN,12911.0,2983537.00,1070214.72,-9735.05,NaN,1444,1444.0,11.84
3,000d7d20,000d7d20_1445,11747.40,NaN,12912.0,2983536.97,1070215.71,-9735.04,104.818205,1445,1445.0,11.81
4,000d7d20,000d7d20_1446,11747.41,NaN,12913.0,2983536.94,1070216.71,-9735.03,103.960888,1446,1446.0,11.78


In [6]:
def rmse(y_true, y_pred) -> float:
    """Compute RMSE in a way that works across scikit-learn versions."""
    try:
        return mean_squared_error(y_true, y_pred, squared=False)
    except TypeError:
        return mean_squared_error(y_true, y_pred) ** 0.5


def make_model(random_state: int = 42) -> HistGradientBoostingRegressor:
    """A simple tabular regression model that handles missing numeric values."""
    return HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.05,
        max_iter=400,
        max_leaf_nodes=31,
        l2_regularization=0.05,
        early_stopping=True,
        validation_fraction=0.1,
        random_state=random_state,
    )


n_splits = min(5, groups.nunique())
fold_scores = []
oof_predictions = np.full(len(model_df), np.nan)

if n_splits >= 2:
    cv = GroupKFold(n_splits=n_splits)

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y, groups), start=1):
        model = make_model(random_state=42 + fold)
        model.fit(X.iloc[train_idx], y.iloc[train_idx])

        valid_pred = model.predict(X.iloc[valid_idx])
        oof_predictions[valid_idx] = valid_pred
        score = rmse(y.iloc[valid_idx], valid_pred)
        fold_scores.append(score)

        print(f"Fold {fold}: RMSE = {score:.5f} | validation wells = {groups.iloc[valid_idx].nunique()}")

    print(f"Mean CV RMSE: {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}")
else:
    print("Not enough wells for grouped validation. Skipping CV and training one model.")

Fold 1: RMSE = 28.53052 | validation wells = 155
Fold 2: RMSE = 22.02040 | validation wells = 155
Fold 3: RMSE = 16.24678 | validation wells = 154
Fold 4: RMSE = 35.70588 | validation wells = 155
Fold 5: RMSE = 21.29194 | validation wells = 154
Mean CV RMSE: 24.75910 +/- 6.72364


## 6. Train Final Model

After validation, train one final model on all non-leaky modeling rows. The model is saved under `models/` so later experiments can compare against this baseline.

In [7]:
final_model = make_model(random_state=2026)
final_model.fit(X, y)

model_path = MODEL_DIR / "baseline_hist_gradient_boosting.joblib"
joblib.dump({"model": final_model, "features": FEATURES}, model_path)

print("Saved model to:", model_path)

Saved model to: C:\Users\alem\Documents\codes\kaggle_rogii\models\baseline_hist_gradient_boosting.joblib


## 7. Generate `submission.csv`

We predict for visible test rows, then merge with `sample_submission.csv` by `id`. This guarantees the output has exactly the required rows, order, and columns.

In [8]:
test_predictions = test_df[["id"]].copy()
test_predictions["tvt"] = final_model.predict(test_df[FEATURES])

submission = sample_submission[["id"]].merge(test_predictions, on="id", how="left")

missing_predictions = submission["tvt"].isna().sum()
if missing_predictions:
    # This should not happen. The fallback prevents an invalid CSV while making the issue visible.
    print(f"Warning: {missing_predictions} submission rows did not match visible test rows. Filling with train target median.")
    submission["tvt"] = submission["tvt"].fillna(y.median())

submission_path = SUBMISSION_DIR / "submission.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)
print("Submission shape:", submission.shape)
display(submission.head())
display(submission.tail())

assert list(submission.columns) == ["id", "tvt"]
assert len(submission) == len(sample_submission)
assert submission["tvt"].notna().all()


Saved submission to: C:\Users\alem\Documents\codes\kaggle_rogii\submissions\submission.csv
Submission shape: (14151, 2)


,id,tvt
0,000d7d20_1442,11752.265589
1,000d7d20_1443,11752.265589
2,000d7d20_1444,11752.265589
3,000d7d20_1445,11752.180743
4,000d7d20_1446,11752.180743


,id,tvt
14146,00e12e8b_6379,11597.540099
14147,00e12e8b_6380,11597.540099
14148,00e12e8b_6381,11597.540099
14149,00e12e8b_6382,11597.540099
14150,00e12e8b_6383,11597.540099


## What to Check After Running

1. The data inspection cells should show the expected train/test counts and that the sample submission has columns `id,tvt`.
2. The modeling rows should be the rows where `TVT_input` is missing in training.
3. The CV output should print one RMSE per fold and a mean RMSE. This is your local baseline score.
4. The final cell should create `submissions/submission.csv` with the same number of rows as `sample_submission.csv` and no missing `tvt` values.
5. Submit `submissions/submission.csv` to Kaggle as the first simple baseline.